# Threat modeling as code with OWASP `pytm`

This notebook is a guided wrapper around `rag_agent_tm.py`. It turns architecture review into a versioned artifact and forces each selected threat to end in a control and a verification method.

In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys

script = Path("01_Threat_Modeling/rag_agent_tm.py")
assert script.exists()
print("Python:", sys.version.split()[0])
print("Model:", script)
print("pytm installed:", importlib.util.find_spec("pytm") is not None)

## 1. Generate or inspect the DFD

The command below asks `pytm` to emit Graphviz DFD source. Graphviz rendering is optional; the text itself is useful in CI and code review.

In [ ]:
if importlib.util.find_spec("pytm"):
    result = subprocess.run(
        [sys.executable, str(script), "--dfd"],
        text=True, capture_output=True, check=False,
    )
    print(result.stdout[:12000])
    if result.stderr:
        print("STDERR:\n", result.stderr[:3000])
    assert result.returncode == 0
else:
    print("Install the pinned workshop dependencies with: pip install -r requirements.txt")

## 2. Derive high-risk paths—not a flat checklist

Complete or extend the rows below. “Verification” must be something engineering can run or inspect, such as a test, permission query, trace assertion, or restore exercise.

In [ ]:
backlog = [
    {
        "unacceptable_outcome": "Secret or cross-tenant data disclosure",
        "attack_path": "Untrusted prompt -> retriever -> mixed-tenant chunk -> model response",
        "prevention": "Tenant filter enforced server-side; source trust metadata; no secrets in prompt context",
        "detection": "Canary and cross-tenant eval suite; retrieval trace sampling",
        "verification": "CI assertion: zero foreign-tenant document IDs in 1,000 adversarial retrievals",
        "owner": "RAG platform",
    },
    {
        "unacceptable_outcome": "Unauthorized refund",
        "attack_path": "Prompt injection -> model emits tool call -> refund API",
        "prevention": "Typed proposal, amount policy, least-privilege token, human approval above INR 500",
        "detection": "Trace decision and approval ID; alert on policy bypass",
        "verification": "Integration test proves no committed refund without a valid approval record",
        "owner": "Agent platform",
    },
    {
        "unacceptable_outcome": "Sensitive data copied to model vendor or logs",
        "attack_path": "User PII -> prompt/trace exporter -> external processor",
        "prevention": "Data minimization and pre-egress redaction",
        "detection": "PII scanner on sampled egress and telemetry",
        "verification": "Known email/phone corpus produces zero raw values in exported spans",
        "owner": "Privacy engineering",
    },
]
backlog

In [ ]:
required = {"unacceptable_outcome", "attack_path", "prevention", "detection", "verification", "owner"}
assert len(backlog) >= 3
assert all(required <= row.keys() and all(str(row[k]).strip() for k in required) for row in backlog)

out = Path("_evidence/01_threat_backlog.json")
out.parent.mkdir(exist_ok=True)
out.write_text(json.dumps(backlog, indent=2), encoding="utf-8")
print("PASS: threat backlog has owners and verification methods")
print("Wrote", out.resolve())

## Review questions

- Which data flow crosses the most consequential trust boundary?
- Which control is enforced outside the model?
- What assumption would make your current threat model false tomorrow?
- What architecture diff should force this file to be reviewed?